# Desarrollo del simulador de interferograma Twyman-Green (modelo por capas)

**Objetivo.** Sustentar, paso a paso, el nuevo simulador `interflib.simulation`: de dónde
sale cada ecuación, por qué se modela cada efecto y cómo se combinan en la imagen final.
No es una reimplementación: cada bloque **importa y disecciona las funciones reales de la
librería**, de modo que este cuaderno queda sincronizado con el código que produce los
interferogramas (regla del puente wiki↔código, `CLAUDE.md §5.4`).

**Idea rectora: separabilidad en tres capas.** Un interferograma real de calibración de un
bloque patrón no es solo franjas con un escalón. Lo construimos como tres capas físicas
independientes, cada una con una responsabilidad única:

| Capa | Qué produce | Física |
|---|---|---|
| **1. Fase óptica** | mapa de diferencia de camino óptico `OPD(x,y)` → fase `φ(x,y)` | tilt de referencia, escalón del bloque, planitud, adherencia, aberración |
| **2. Formación de imagen** | irradiancia continua `I(x,y)` | `I = I₀(1 + V cos φ)`, iluminación, visibilidad, PSF |
| **3. Sensor** | imagen digital en cuentas | ruido de disparo y lectura, ganancia, cuantización, saturación |

Esta separación es la decisión de arquitectura central (ADR-005): permite razonar,
probar y calibrar cada efecto por separado, y deja *ganchos* limpios para los modos
multi-λ y phase-shifting que se añadirán después.

**Convenciones de unidades y signo.** Longitud de onda `λ` y todo `OPD`/altura en
**nanómetros**; campo de visión y bloque en **milímetros**. Doble paso: una altura de
superficie `h` produce `OPD = 2h` y una fase `φ = (2π/λ)·OPD`. Símbolos según
[`wiki/nomenclatura.md`].


## 0. Preparación

Importamos la API pública del simulador y, además, sus **funciones internas** (prefijo
`_`): son las que arma la librería capa por capa y las usaremos para mostrar cada
contribución por separado. Trabajar con ellas (y no con copias) es lo que garantiza que
lo explicado aquí es exactamente lo que se ejecuta.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

from interflib.simulation import (
    SceneGeometry, PhaseModel, Illumination, SensorModel,
    synthesize_interferogram, fringe_fraction_of_length,
)
# Internos de la librería que diseccionamos capa por capa (única fuente de verdad):
from interflib.simulation.interferogram import (
    _coordinate_grids, _block_mask, _tilt_height_nm, _low_order_flatness_nm,
    _aberration_opd_nm, _illumination, _visibility, _apply_sensor,
)

plt.rcParams.update({"figure.dpi": 110, "image.cmap": "gray", "font.size": 10})

def show(img, title="", cmap="gray", colorbar=False, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(4.2, 4.2))
    im = ax.imshow(img, extent=[-1, 1, -1, 1], cmap=cmap)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
    if colorbar:
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    return ax

## 1. La ecuación fundamental: interferencia de dos haces

En un Twyman-Green la luz colimada se divide en dos brazos (referencia y prueba) y se
recombina [`wiki/conceptos/interferometro-twyman-green.md`]. Sumando dos campos escalares
monocromáticos de amplitudes $a_1, a_2$ y fases $\varphi_1, \varphi_2$,

$$E = a_1 e^{i\varphi_1} + a_2 e^{i\varphi_2}, \qquad
I = |E|^2 = a_1^2 + a_2^2 + 2 a_1 a_2 \cos(\varphi_1 - \varphi_2).$$

Con $I_1 = a_1^2$, $I_2 = a_2^2$ y $\Delta\phi = \varphi_1 - \varphi_2$ se reescribe en la
forma canónica de **dos haces** [`hecht-2017-optics`]:

$$\boxed{\,I = I_0\,\bigl(1 + V\cos\Delta\phi\bigr)\,},\qquad
I_0 = I_1 + I_2,\qquad V = \frac{2\sqrt{I_1 I_2}}{I_1 + I_2}.$$

$I_0$ es la **irradiancia de fondo** y $V$ el **contraste o visibilidad**
($V = (I_\max - I_\min)/(I_\max + I_\min)$). Con haces iguales $V = 1$; la coherencia
parcial, el desbalance de intensidades o la mala superposición lo reducen (capa 2). Toda
la información métrica está en $\Delta\phi$.

In [ ]:
phi = np.linspace(0, 6 * np.pi, 500)
fig, ax = plt.subplots(figsize=(7, 3))
for V in (1.0, 0.6, 0.3):
    ax.plot(phi / np.pi, 0.5 * (1 + V * np.cos(phi)), label=f"V = {V}")
ax.set_xlabel(r"$\Delta\phi/\pi$"); ax.set_ylabel("I  (I0 = 0.5)")
ax.set_title(r"$I = I_0\,(1 + V\cos\Delta\phi)$"); ax.legend()
plt.show()

### El factor de doble paso

En el brazo de prueba la luz **va y vuelve** de la superficie: un desnivel $h$ recorre un
camino extra $2h$. Por eso la diferencia de camino óptico y la fase son

$$\mathrm{OPD} = 2h, \qquad \Delta\phi = \frac{2\pi}{\lambda}\,\mathrm{OPD}
= \frac{4\pi}{\lambda}\,h.$$

Este factor 2 es sutil y **es la corrección clave frente al prototipo anterior**, que lo
omitía al contar franjas (lo veremos en §3.1) [`malacara-2007-optical-shop-testing`,
`wiki/conceptos/interferometro-twyman-green.md`].

**Lectura de la figura.** Cada curva es un corte 1-D de la intensidad al barrer la fase.
La media (`I0`) es la misma en las tres; lo que cambia con `V` es la **amplitud de la
modulación**: con `V=1` la intensidad va de 0 a `2·I0` (contraste total), y al bajar `V` las
franjas se aplanan hacia el valor medio. Esta es, en 1-D, la pérdida de contraste que verás en
las imágenes cuando la coherencia o la superposición no sean ideales; fija la intuición de qué
significa `V` antes de meterlo en 2-D.

## 2. La escena: bloque adherido a la platina

El bloque patrón se adhiere (*wringing*) a un plato de referencia; ambos se comparan
contra el mismo plano de referencia, así que **comparten el mismo tilt**. Lo observable es
la **discontinuidad de las franjas** en el borde del bloque
[`wiki/conceptos/interferometro-twyman-green.md`].

La geometría (campo de visión, huella del bloque, posición y rotación) vive en
`SceneGeometry`. La huella se calcula con `_block_mask`, que sitúa el rectángulo en
coordenadas físicas (mm) y admite descentrado y rotación en el plano.

In [ ]:
geom = SceneGeometry(resolution=512, fov_mm=25.0, block_width_mm=9.0, block_height_mm=40.0)
# Dos sistemas de coordenadas (ver 2.1): por eje (Xn,Yn) e isótropo (Xr,Yr).
Xn, Yn, Xr, Yr = _coordinate_grids(geom)
mask = _block_mask(geom, Xn, Yn)

fig, ax = plt.subplots(1, 2, figsize=(8.2, 4.2))
show(mask, "Máscara del bloque (9 x 40 mm en 25 mm)", ax=ax[0])
mask_rot = _block_mask(SceneGeometry(fov_mm=25, block_width_mm=9, block_height_mm=20,
                                     block_center_mm=(3, 0), block_rotation_deg=12), Xn, Yn)
show(mask_rot, "Descentrado + rotado (para probar ROIs)", ax=ax[1])
plt.tight_layout(); plt.show()

**Lectura de la figura.** Esto **no** es una imagen física, es la geometría: blanco = dentro
del bloque, negro = platina. El panel izquierdo es un bloque centrado de 9×40 mm; el derecho
muestra que la huella admite descentrado y rotación. Esta máscara es la que decide dónde se
inyecta el escalón de longitud y, más adelante, marca las **regiones de interés (ROI)** que el
lector de fracción promedia por separado (bloque vs. platina). Verla aparte evita confundir la
geometría con los efectos ópticos que vienen después.

### 2.1 Dos sistemas de coordenadas (y sensores no cuadrados)

`_coordinate_grids` devuelve **dos** pares de mallas, porque no todos los efectos escalan
igual:

- **Por eje** `(Xn, Yn)`, normalizadas a `[-1, 1]` en cada eje. Rigen lo que se define *por
  eje*: el **conteo de franjas** del tilt (`fringes_x` de borde a borde en x, `fringes_y` en y)
  y la **máscara del bloque** (que usa el campo de visión de cada eje).
- **Isótropas** `(Xr, Yr)`, que comparten **una sola escala física**. Rigen los efectos
  **radiales y circulares** (haz, viñeteo, aberración de frente de onda, planitud): un haz
  circular debe verse circular, no estirarse con el marco.

En un sensor **cuadrado** ambos pares coinciden. En uno **no cuadrado** (como la CS505MU,
2448×2048), `Yr = Yn·(h−1)/(w−1)`, lo que mantiene el **paso de píxel idéntico en los dos
ejes** (píxeles cuadrados) y deja los círculos circulares. `SceneGeometry` acepta
`resolution` como entero (cuadrado) o `(alto, ancho)`; el preset `SceneGeometry.cs505mu(scale=…)`
arma la relación de aspecto del sensor real.

In [ ]:
# Geometría de la CS505MU (aspecto real 2448:2048), reducida para que sea ligera.
geom_cam = SceneGeometry.cs505mu(scale=0.22, fov_mm=25.0, block_width_mm=9.0, block_height_mm=40.0)
print("resolución (alto, ancho):", geom_cam.resolution)

cam = synthesize_interferogram(
    geom_cam,
    PhaseModel(wavelength_nm=632.8, central_length_nm=158.2, fringes_x=0.4, fringes_y=6.0,
               defocus_nm=100.0),
    Illumination(background=0.5, beam_waist_frac=0.8, vignette=0.5,
                 visibility=0.85, psf_sigma_px=0.8),
    SensorModel.cs505mu(), seed=1)
print("imagen:", cam.image.shape, cam.image.dtype)

fig, ax = plt.subplots(figsize=(6.2, 5.2))
ax.imshow(cam.image, cmap="gray"); ax.set_title("CS505MU: marco no cuadrado, viñeteo circular")
ax.set_xticks([]); ax.set_yticks([]); plt.show()

**Lectura de la figura.** La imagen ya no es cuadrada: su relación de aspecto es 2448:2048,
la del sensor. Lo importante es que el **viñeteo (el oscurecimiento) es circular**, no elíptico,
aunque el marco sea rectangular; eso es lo que garantizan las coordenadas isótropas. Si el haz
usara coordenadas por eje, el círculo se estiraría con el marco (un artefacto que no ocurre en
un sistema óptico real). Esta celda demuestra que geometría no cuadrada y realismo del haz son
compatibles.

### El mensurando y lo observable: la fracción de franja

La longitud central $L$ del bloque introduce un salto de fase entre las franjas del bloque
y las de la platina:

$$\Delta\phi_{\text{bloque}} = \frac{4\pi}{\lambda} L.$$

Pero el coseno es periódico: solo se observa esa fase **módulo $2\pi$**, es decir la
**fracción de franja**

$$F = \operatorname{frac}\!\left(\frac{2L}{\lambda}\right) \in [0, 1),$$

con rango no ambiguo $\lambda/2$. El número entero de órdenes $\kappa$ se resuelve aparte
(método de las **fracciones exactas**, multi-λ) [`wiki/conceptos/metodo-fracciones-exactas.md`,
`hariharan-2003-optical-interferometry`]. El simulador inyecta $F$ como verdad de terreno
y devuelve `fraction_truth`; la función `fringe_fraction_of_length` la calcula:

In [ ]:
lam = 632.8  # nm (He-Ne rojo)
for L in (79.1, 158.2, 316.4):
    print(f"L = {L:6.1f} nm  ->  2L/λ = {2*L/lam:6.3f}  ->  F = {fringe_fraction_of_length(L, lam):.3f}")

### El escalón se inyecta *reducido*

Un bloque real mide milímetros: $L \sim 10^8$ nm. Meter ese número en el $\cos$ es
numéricamente frágil y, sobre todo, **innecesario**: solo la fracción es observable. Por
eso el simulador añade en la región del bloque el escalón reducido
$F\cdot\lambda$ (en OPD), más un término opcional por **cambio de fase en la reflexión**
$\varphi_\text{bloque}-\varphi_\text{platina}$ [`wiki/conceptos/correccion-cambio-fase.md`],
que por defecto es 0 para casar con el pipeline EF actual (que aún no aplica esa
corrección).

## 3. Capa 1 — El mapa de fase óptico

Construimos $\mathrm{OPD}(x,y)$ sumando contribuciones, cada una con su justificación.

### 3.1 Tilt de referencia (la portadora de franjas) y el arreglo del ×2

Inclinar el plano de referencia produce una rampa lineal de OPD → franjas rectas y
paralelas. ¿Cuántas franjas? Una franja equivale a un cambio de OPD de $\lambda$. Para
tener $N$ franjas de borde a borde:

$$\Delta(\mathrm{OPD}) = N\lambda \;\Rightarrow\; \Delta h = \frac{N\lambda}{2}
\quad(\text{porque } \mathrm{OPD}=2h).$$

Con $X\in[-1,1]$, `_tilt_height_nm` fija $h(X)=\tfrac{N\lambda}{2}\cdot\tfrac{X}{2}$, cuyo
recorrido es $N\lambda/2$ → exactamente $N$ franjas. **El prototipo anterior olvidaba el
$\tfrac12$ del doble paso y producía $2N$ franjas.** Lo comprobamos contando cruces por
cero del coseno a lo largo de una columna:

In [ ]:
h_tilt = _tilt_height_nm(Xn, Yn, fringes_x=0.0, fringes_y=8.0, wavelength_nm=lam)
phi_tilt = (2 * np.pi / lam) * (2 * h_tilt)        # φ = (2π/λ)·OPD,  OPD = 2h

col = np.cos(phi_tilt[:, 256])
n_fringes = np.sum(np.diff(np.signbit(col))) / 2
show(np.cos(phi_tilt), f"Tilt vertical: {n_fringes:.0f} franjas (pedidas 8)")
plt.show()

**Lectura de la figura.** Franjas rectas y paralelas: es la **portadora** que introduce el
tilt del plano de referencia. Sin ella, un bloque sin inclinación daría un campo casi uniforme,
imposible de leer. El conteo impreso de cruces por cero confirma que hay exactamente **8**
franjas (las pedidas), no 16: esa es la comprobación numérica del factor de doble paso
(`OPD = 2h`) que discutimos en §1; si el modelo se equivocara en ×2, aquí saldrían 16.

### 3.2 Escalón del bloque

Dentro de la máscara, la fase salta $2\pi F$. Dos longitudes distintas dan dos fracciones
distintas; con $F=0.5$ (media franja) la franja del bloque queda **invertida** respecto a
la platina, y con $F\to 0$ o $F\to 1$ se alinea con ella.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4.2))
for ax, L in zip(axes, (0.0, 79.1, 158.2)):   # F = 0.00, 0.25, 0.50
    r = synthesize_interferogram(
        geom, PhaseModel(wavelength_nm=lam, central_length_nm=L, fringes_y=8.0),
        Illumination(psf_sigma_px=0.0), SensorModel(shot_noise=False, read_noise_e=0.0),
        seed=0)
    show(r.image, f"L = {L} nm   →   F = {r.fraction_truth:.2f}", ax=ax)
plt.tight_layout(); plt.show()

**Lectura de la figura.** Las tres imágenes comparten la misma portadora; solo cambia `L`.
Fíjate en la **franja central (la del bloque)** frente a las de la platina: con `F=0` continúan
alineadas; con `F=0.5` (media franja) el bloque queda **en oposición** (sus máximos donde la
platina tiene mínimos); con `F=0.25` hay un desfase intermedio. Ese **salto lateral de las
franjas en el borde del bloque es, literalmente, la medida**: la fracción `F`. Toda la
metrología de longitud por interferometría se reduce a leer ese salto con precisión.

### 3.3 Planitud de superficies

Ninguna superficie es un plano perfecto. Modelamos la desviación de planitud del bloque y
de la platina como un término suave de bajo orden ("bol"), parametrizado por su
valor pico-valle (PV, en nm). Se traduce en franjas ligeramente curvas sobre cada
superficie [`malacara-2007-optical-shop-testing`].

In [ ]:
flat = _low_order_flatness_nm(Xr, Yr, pv_nm=200.0)
show(flat, "Planitud: bol de 200 nm PV", cmap="viridis", colorbar=True)
plt.show()

**Lectura de la figura.** El mapa (en nm, escala *viridis*) es la desviación de planitud
modelada como un 'bol' suave de bajo orden. **No** se ve como franjas porque es una contribución
de **altura**, no de intensidad; su efecto llega después, curvando ligeramente las franjas sobre
la superficie. El PV de 200 nm es exagerado a propósito para que se aprecie; las superficies de
un patrón real están muy por debajo. Lo generamos para poder inyectar imperfección de superficie
controlada y ver cuánto tolera el ajuste.

### 3.4 Desajuste de tilt por adherencia (*wringing*)

La adherencia no deja el bloque perfectamente paralelo a la platina: aparece un pequeño
tilt **adicional solo sobre el bloque**. Se reutiliza `_tilt_height_nm` con la máscara,
con los parámetros `block_tilt_mismatch_x/y`. Efecto: las franjas del bloque no son la
mera continuación de las de la platina.

### 3.5 Aberración de frente de onda de la óptica

Colimación y planos imperfectos introducen un error de frente de onda $W(x,y)$ sobre todo
el campo. Lo modelamos con los términos de Zernike de bajo orden más comunes
[`goodwin-2006-field-guide-testing`]:

$$W \approx a_\text{def}(2\rho^2-1) + a_\text{ast}(x^2 - y^2) + a_\text{coma}(3\rho^2-2)\,y,
\qquad \rho^2 = x^2 + y^2,$$

con los coeficientes en nm. A diferencia del tilt/planitud (superficie, doble paso), la
aberración es un error de **frente de onda** que entra como OPD de **un solo paso**. Su
efecto es una curvatura global de las franjas.

In [ ]:
ab = _aberration_opd_nm(Xr, Yr, PhaseModel(defocus_nm=150.0, astigmatism_nm=100.0, coma_nm=80.0))
fig, ax = plt.subplots(1, 2, figsize=(8.6, 4.2))
show(ab, "Aberración: OPD de frente de onda [nm]", cmap="viridis", colorbar=True, ax=ax[0])
phi_ab = (2 * np.pi / lam) * ((2 * _tilt_height_nm(Xn, Yn, 0, 8, lam)) + ab)
show(np.cos(phi_ab), "Franjas curvadas por la aberración", ax=ax[1])
plt.tight_layout(); plt.show()

**Lectura de la figura.** Izquierda: el error de frente de onda de la óptica (OPD en nm),
aquí desenfoque + astigmatismo + coma. Derecha: las **mismas** franjas del tilt, pero ahora
**curvadas** por esa aberración. Distinguir esto es clave en un interferómetro real: franjas que
no son perfectamente rectas suelen delatar la **calidad de la óptica** o un desenfoque, **no** un
defecto del bloque. Lo simulamos para que el pipeline aprenda a atravesar franjas curvas sin
confundirlas con la señal de longitud.

### 3.6 Ensamble de la capa 1

Sumando todo: $\mathrm{OPD} = 2h_\text{superficie} + W$, con $h_\text{superficie}$ = tilt +
planitud (+ desajuste y planitud del bloque, dentro de la máscara), y el escalón reducido
en la región del bloque. La fase es $\phi = (2\pi/\lambda)\,\mathrm{OPD}$. El simulador
devuelve el mapa OPD y la fase envuelta como verdad de terreno.

In [ ]:
phase = PhaseModel(wavelength_nm=lam, central_length_nm=158.2, fringes_x=0.5, fringes_y=8.0,
                   block_tilt_mismatch_x=0.4, block_tilt_mismatch_y=0.3,
                   block_flatness_pv_nm=60.0, platen_flatness_pv_nm=40.0,
                   defocus_nm=120.0, astigmatism_nm=80.0, coma_nm=50.0)
r = synthesize_interferogram(geom, phase, Illumination(psf_sigma_px=0.0),
                             SensorModel(shot_noise=False, read_noise_e=0.0), seed=0)
fig, ax = plt.subplots(1, 3, figsize=(12.6, 4.2))
show(r.opd_nm, "OPD total [nm]", cmap="viridis", colorbar=True, ax=ax[0])
show(r.phase, "Fase envuelta φ [rad]", cmap="twilight", colorbar=True, ax=ax[1])
show(np.cos(r.phase), "cos φ (franjas ideales)", ax=ax[2])
plt.tight_layout(); plt.show()

**Lectura de la figura.** Las tres vistas son el **mismo** mapa, presentado de tres formas
que conviene no confundir:

- **OPD total (izq.):** la diferencia de camino óptico en **nm**, continua y sin envolver.
  Domina la rampa del tilt (de ahí el degradado suave de un extremo a otro); el escalón del
  bloque, la planitud y la aberración son pequeños encima de esa rampa. Es la 'verdad' física en
  unidades de longitud, la que ningún instrumento observa directamente.

- **Fase envuelta φ (centro):** la fase `φ = (2π/λ)·OPD`, pero **plegada a `(−π, π]`** con
  `np.angle(exp(iφ))`. **¿Por qué se ve a bandas con saltos bruscos?** Porque la fase real da
  **una vuelta completa de 2π por cada franja** (aquí ~8 en vertical): al plegarla, cada vez que
  cruza un múltiplo impar de π 'salta' de +π a −π, y ese salto es la línea dura que ves. No es un
  error: es exactamente la información que sobrevive a la medición. **¿Por qué generarla?** Porque
  es lo **único observable**: la intensidad depende de `cos φ`, que solo distingue la fase
  **módulo 2π**; el número entero de vueltas (la parte 'grande' de la OPD) se pierde, y esa es
  justo la **ambigüedad de orden entero** que resuelve el método de fracciones exactas. **¿Qué
  aporta?** Es el punto de partida literal de los algoritmos de **phase-shifting / unwrapping**
  (línea L2): a partir de varias imágenes recuperan esta fase envuelta y luego intentan
  'desenvolverla'. Tenerla aquí como verdad de terreno permite comprobar esos algoritmos.

- **cos φ (der.):** el coseno de esa fase → el patrón de franjas **ideal** (sin ruido ni
  iluminación). Cada franja brillante es un contorno donde `φ` es múltiplo de 2π; dicho de otro
  modo, **las franjas son las líneas donde la fase envuelta cruza por cero**. Es el puente entre
  el mapa de fase y lo que registrará la cámara.

En una frase: **OPD** (físico, continuo) → **φ envuelta** (observable, módulo 2π, base del
unwrapping) → **cos φ** (las franjas). Las tres capas siguientes solo *ensucian* esta última.

## 4. Capa 2 — Formación de la imagen

Ya tenemos la fase. La irradiancia continua es $I(x,y) = I_0(x,y)\,[1 + V(x,y)\cos\phi]$,
donde ahora $I_0$ y $V$ **varían en el campo**.

### 4.1 Iluminación no uniforme $I_0(x,y)$

Un haz real tiene perfil aproximadamente gaussiano y la óptica introduce viñeteo (caída
radial hacia los bordes). `_illumination` combina un núcleo gaussiano de cintura
`beam_waist_frac` con un término de viñeteo.

In [ ]:
i0 = _illumination(Xr, Yr, Illumination(background=0.5, beam_waist_frac=1.1, vignette=0.35))
show(i0, "Iluminación de fondo I0(x,y)", cmap="magma", colorbar=True)
plt.show()

**Lectura de la figura.** El fondo `I0(x,y)`: brillante en el centro (núcleo gaussiano del
haz) y oscurecido hacia los bordes (viñeteo). Es la **envolvente que multiplica** a las franjas,
así que en la imagen final el brillo y el contraste caen hacia las esquinas. Por eso el pipeline
tiene que **estimar y quitar este fondo** antes de leer la fase: si no, confundiría la caída de
iluminación con una variación de la señal.

### 4.2 Visibilidad $V(x,y)$

La visibilidad baja de 1 por coherencia parcial, desbalance de intensidades o
superposición imperfecta, y suele **decaer hacia los bordes**
[`hecht-2017-optics`, `wiki/conceptos/interferometro-twyman-green.md`]. `_visibility` da un
$V_0$ base con una caída radial opcional.

In [ ]:
vis = _visibility(Xr, Yr, Illumination(visibility=0.85, visibility_falloff=0.3))
show(vis, "Mapa de visibilidad V(x,y)", cmap="cividis", colorbar=True)
plt.show()

**Lectura de la figura.** El contraste local `V(x,y)`: alto en el centro, decae hacia los
bordes. Ojo a la diferencia con `I0`: mientras `I0` sube o baja el **brillo medio**, `V` controla
la **profundidad** de las franjas. Donde `V` es bajo, las franjas casi desaparecen **aunque haya
luz de sobra** — por eso `V` e `I0` son efectos distintos y se modelan por separado.

### 4.3 Borrosidad por PSF (MTF finita)

La óptica de imagen tiene resolución finita: cada punto se difumina según su función de
dispersión de punto (PSF). Lo aproximamos con un filtrado gaussiano de sigma
`psf_sigma_px` (píxeles). Suaviza las franjas y baja el contraste de las más finas.

In [ ]:
sharp = i0 * (1 + vis * np.cos(phi_ab))
blur = gaussian_filter(sharp, 1.6)
fig, ax = plt.subplots(1, 2, figsize=(8.6, 4.2))
show(sharp, "Sin PSF (sigma = 0)", ax=ax[0])
show(blur, "Con PSF (sigma = 1.6 px)", ax=ax[1])
plt.tight_layout(); plt.show()

**Lectura de la figura.** Izquierda, franjas nítidas; derecha, tras la PSF (la borrosidad
de la óptica de imagen). El desenfoque **reduce más el contraste de las franjas finas** que el de
las gruesas: la MTF cae con la frecuencia espacial. Importa porque en la práctica **limita
cuántas franjas por milímetro se pueden resolver** — meter demasiadas franjas y una PSF ancha
degrada la lectura de fase.

## 5. Capa 3 — El sensor

La cámara convierte irradiancia en cuentas digitales (ADU). Modelamos la cadena estándar
de un sensor [modelo tipo EMVA 1288; `[sin fuente — modelo estándar de sensor]`]:

1. **Fotoelectrones**: la irradiancia (normalizada a 1 = saturación) se escala a un número
   medio de electrones (`full_well_e`).
2. **Ruido de disparo (Poisson)**: los fotones llegan aleatoriamente; la varianza en
   electrones **es igual a la media** ($\sigma_e = \sqrt{N_e}$), así que el ruido **crece
   con la señal**.
3. **Ruido de lectura (gaussiano)**: aditivo, independiente de la señal.
4. **Ganancia y bias**: electrones → ADU, más un offset oscuro.
5. **Cuantización**: se redondea a la profundidad de bits (8/12/16).
6. **Saturación** al fondo de escala; opcionalmente **píxeles calientes** y **polvo**.

Lo comprobamos con una rampa de irradiancia: la $\sigma$ por columna debe crecer como
$\sqrt{\text{señal}}$ (firma del ruido de disparo).

In [ ]:
ramp = np.tile(np.linspace(0.02, 0.98, 512), (512, 1))
rng = np.random.default_rng(0)
noisy = _apply_sensor(ramp, SensorModel(bit_depth=8, full_well_e=6000.0,
                                        read_noise_e=6.0, shot_noise=True), rng)
col_std = noisy.astype(float).std(axis=0)
fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.4))
ax[0].imshow(noisy, cmap="gray"); ax[0].set_title("Rampa de irradiancia + sensor"); ax[0].axis("off")
ax[1].plot(np.linspace(0.02, 0.98, 512), col_std)
ax[1].set_xlabel("irradiancia (0..1)"); ax[1].set_ylabel(r"$\sigma$ por columna [ADU]")
ax[1].set_title(r"El ruido crece con la señal ($\sqrt{N}$)")
plt.tight_layout(); plt.show()

**Lectura de la figura.** Izquierda: una rampa de irradiancia (oscuro→claro) pasada por el
sensor. Derecha: la desviación estándar **por columna** crece con la señal siguiendo ~√(señal),
la **firma inconfundible del ruido de disparo** (Poisson). En el extremo oscuro (izquierda de la
curva) la σ no baja a cero sino a un **piso plano**: ahí domina el **ruido de lectura**, que es
constante. Esto explica por qué las zonas oscuras de un interferograma se ven, en términos
relativos, más 'sucias', y por qué conviene trabajar con las franjas bien expuestas.

## 6. Cierre del lazo: la verdad de terreno se recupera

La prueba definitiva de que el simulador es coherente: si generamos una imagen con una
longitud $L$ conocida y la pasamos por el lector de fracción de franja de la librería
(`FringeFractionMeasurement`, [`wiki/conceptos/analisis-de-franjas.md`]), debemos recuperar
`fraction_truth`. Esto es lo que ancla el simulador al resto del pipeline (y es un test
automático de la suite).

In [ ]:
from interflib.fringe.profile_fit import FringeFractionMeasurement

L = 79.1  # 2L/λ = 0.25 exacto -> fracción no ambigua
clean = synthesize_interferogram(
    SceneGeometry(resolution=512, fov_mm=25.0, block_width_mm=9.0, block_height_mm=40.0),
    PhaseModel(wavelength_nm=lam, central_length_nm=L, fringes_y=8.0),
    Illumination(background=0.5, beam_waist_frac=1e6, vignette=0.0,
                 visibility=0.9, visibility_falloff=0.0, psf_sigma_px=0.5),
    SensorModel(bit_depth=16, shot_noise=False, read_noise_e=0.0),
    seed=0)

roi_platen = (60, 452, 40, 130)   # (y1, y2, x1, x2)  a la izquierda del bloque
roi_block = (60, 452, 220, 300)   # franja central del bloque
measured = FringeFractionMeasurement(clean.image, roi_platen, roi_block)\
    .measure(pixels_between_fringes=512 / 8).fringe_fraction

print(f"F inyectada  : {clean.fraction_truth:.4f}")
print(f"F recuperada : {measured:.4f}")
print(f"error        : {abs(measured - clean.fraction_truth):.4f}")

**Lectura del resultado.** No hay gráfica: hay tres números. La `F` **inyectada** por el
simulador y la `F` **recuperada** por el lector coinciden (error 0,0000 en el caso limpio). Es la
prueba de que el desfase que el simulador mete en la región del bloque es exactamente el que el
pipeline lee de vuelta: **el lazo cierra**. Por eso este simulador sirve como banco de pruebas —
si mañana un cambio en el análisis rompe la recuperación, este número deja de dar cero.

## 7. Una imagen realista con la cámara del INM

Combinando las tres capas obtenemos un interferograma que se parece a los reales: franjas
curvas por aberración, contraste que cae hacia los bordes, viñeteo, ruido y polvo. Aquí la
capa 3 y la geometría son las de la **cámara real del INM**: `SensorModel.cs505mu()` (12 bits,
full well ≥10 ke⁻, ruido de lectura <2,5 e⁻) y `SceneGeometry.cs505mu()` (marco no cuadrado
2448×2048). Añadimos polvo y píxeles calientes sobre el preset de datasheet (son defectos
extra, no parte de la especificación).

In [ ]:
# Sensor = preset de la CS505MU, más defectos extra (polvo, píxeles calientes).
cam = SensorModel.cs505mu()
cam.hot_pixel_fraction = 2e-5
cam.dust_specks = 4
cam.dust_radius_px = 7.0

real = synthesize_interferogram(
    SceneGeometry.cs505mu(scale=0.3, fov_mm=25.0, block_width_mm=9.0, block_height_mm=40.0),
    PhaseModel(wavelength_nm=lam, central_length_nm=158.2, fringes_x=0.6, fringes_y=8.0,
               block_tilt_mismatch_x=0.4, block_tilt_mismatch_y=0.3,
               block_flatness_pv_nm=60.0, platen_flatness_pv_nm=40.0,
               defocus_nm=120.0, astigmatism_nm=80.0, coma_nm=50.0),
    Illumination(background=0.5, beam_waist_frac=1.1, vignette=0.35,
                 visibility=0.8, visibility_falloff=0.25, psf_sigma_px=1.2),
    cam, seed=7)
print("imagen:", real.image.shape, real.image.dtype)   # (alto, ancho), 12 bits -> uint16

# imshow directo (no show()): preserva el aspecto no cuadrado del sensor.
fig, ax = plt.subplots(figsize=(5.6, 4.9))
ax.imshow(real.image, cmap="gray")
ax.set_title("Interferograma realista — cámara CS505MU (12 bits, 2448:2048)")
ax.set_xticks([]); ax.set_yticks([]); plt.show()

**Lectura de la figura.** La imagen culminante junta todo: franjas **curvadas** (aberración),
**contraste y brillo que caen** hacia los bordes (`I0`+`V`+viñeteo), **ruido** de la CS505MU a 12
bits, y algún **píxel caliente o mota de polvo**. Compárala mentalmente con el `cos φ` ideal del
§3.6: esa es exactamente **la distancia entre 'la física' y 'lo que entrega la cámara'**, y es el
terreno que el análisis de franjas debe atravesar para recuperar la fracción `F`. Todo lo que
añadimos de §4 en adelante fue precisamente cerrar esa distancia.

### Parámetros del modelo

| Dataclass | Controla | Parámetros clave |
|---|---|---|
| `SceneGeometry` | geometría | `resolution` (entero cuadrado o `(alto, ancho)`), `fov_mm`, `block_width_mm`, `block_height_mm`, `block_center_mm`, `block_rotation_deg`; preset `SceneGeometry.cs505mu(scale, fov_mm)` |
| `PhaseModel` | capa 1 (fase) | `wavelength_nm`, `central_length_nm`, `fringes_x/y`, `block_tilt_mismatch_x/y`, `block/platen_flatness_pv_nm`, `defocus/astigmatism/coma_nm`, `phase_change_diff_rad`, `phase_shift_rad` |
| `Illumination` | capa 2 (imagen) | `background`, `beam_waist_frac`, `vignette`, `visibility`, `visibility_falloff`, `psf_sigma_px` |
| `SensorModel` | capa 3 (sensor) | `bit_depth`, `full_well_e`, `read_noise_e`, `dark_offset_adu`, `shot_noise`, `saturate`, `hot_pixel_fraction`, `dust_specks`; preset `SensorModel.cs505mu()` (cámara real del INM) |

**Ganchos para lo diferido.** `phase_shift_rad` (un offset global de fase) es el gancho
para armar **stacks phase-shifting** (varios cuadros con pasos conocidos, línea L2); y como
`PhaseModel` está parametrizado por `wavelength_nm`, un **conjunto multi-λ** con la misma
geometría (para el método de fracciones exactas) es un bucle sobre λ sin tocar la física.


## 8. Referencias

**Wiki**
- [`wiki/conceptos/interferometro-twyman-green.md`] — montaje y principio.
- [`wiki/conceptos/metodo-fracciones-exactas.md`] — orden entero y fracción de franja.
- [`wiki/conceptos/analisis-de-franjas.md`] — lectura de la fracción (el lado inverso).
- [`wiki/conceptos/correccion-cambio-fase.md`] — cambio de fase en la reflexión.
- [`wiki/nomenclatura.md`] — símbolos y unidades.
- [`wiki/metodos/simulacion-interferograma.md`] — página de método (ancla del código).
- [`wiki/decisiones/ADR-005-simulador-interferograma.md`] — decisión de arquitectura.
- [`wiki/instrumentos/interferometro-inm.md`] — el interferómetro y su cámara CS505MU.

**Fuentes**
- [`hecht-2017-optics`] — interferencia de dos haces, visibilidad y coherencia.
- [`malacara-2007-optical-shop-testing`] — Twyman-Green, doble paso, superficies.
- [`hariharan-2003-optical-interferometry`] — medición de longitud por fracciones exactas.
- [`goodwin-2006-field-guide-testing`] — polinomios de Zernike y aberraciones.
- [`datasheet-thorlabs-cs505mu`] — specs del sensor (12 bits, full well, ruido de lectura); fuente del preset `SensorModel.cs505mu()`.

**Marcas de trazabilidad.** La *estructura* del modelo de sensor (ruido de disparo/lectura,
cuantización) sigue la práctica estándar de sensores (tipo EMVA 1288), sin fuente propia en el
wiki → `[sin fuente — modelo estándar de sensor]`. En cambio, los **valores del preset
`SensorModel.cs505mu()`** sí están anclados a la datasheet [`datasheet-thorlabs-cs505mu`]
(12 bits, full well ≥10 000 e⁻, ruido de lectura <2,5 e⁻). Los defaults **genéricos** de
`SensorModel`/`Illumination` siguen siendo ilustrativos y ajustables, no afirmaciones
metrológicas.
